# Helios Cloud GPU Pretraining (resume to 2B tokens)

Run the cells top to bottom. Safe to re-run after any disconnect: weights, tokenizer and the dedup ledger live in Google Drive and Hugging Face, so training resumes where it stopped and never repeats a record.

**Before cell 3:** add your Hugging Face token under the key icon (Secrets) as `HF_TOKEN` and enable notebook access.

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > A100 or H100"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/2**30:.0f} GB", "| bf16:", torch.cuda.is_bf16_supported())

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/Helios')
for d in ('checkpoints', 'manifests', 'datasets', 'tokenizer'):
    (DRIVE_ROOT / d).mkdir(parents=True, exist_ok=True)
print("Drive root:", DRIVE_ROOT)

## 3. Get the latest code and your Hugging Face token

In [ ]:
import os
%cd /content
!test -d helios_repo || git clone -b pretrained https://github.com/darknecrocities/Helios.git helios_repo
%cd /content/helios_repo
!git pull origin pretrained
!pip install -q -e . safetensors rich tokenizers datasets huggingface_hub

from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
except Exception as e:
    raise RuntimeError("Add HF_TOKEN in Colab Secrets (key icon) and enable notebook access, then re-run.") from e
print("HF_TOKEN loaded")

## 4. Fetch corpus, ledger and trained tokenizer
Only downloads what is missing from Drive. The ledger is the record of already-trained data, so keep it intact.

In [ ]:
import json, shutil
from huggingface_hub import hf_hub_download
tok = os.environ["HF_TOKEN"]

def fetch(repo, name, dest, repo_type):
    if not dest.exists():
        print("Downloading", name)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(hf_hub_download(repo, name, repo_type=repo_type, token=tok), dest)

fetch('Darknecrocities/Helios-Pretrain-Corpus', 'helios_massive_pretrain_corpus.jsonl', DRIVE_ROOT/'datasets'/'helios_massive_pretrain_corpus.jsonl', 'dataset')
fetch('Darknecrocities/Helios-Pretrain-Corpus', 'curriculum_ledger.json', DRIVE_ROOT/'manifests'/'curriculum_ledger.json', 'dataset')
for f in ('tokenizer.json', 'special_tokens_map.json'):
    fetch('Darknecrocities/Helios', f'tokenizer/{f}', DRIVE_ROOT/'tokenizer'/f, 'model')

ledger = json.load(open(DRIVE_ROOT/'manifests'/'curriculum_ledger.json'))
print("Resume cursor:", ledger['category_cursors']['all'], "| records already trained:", len(ledger['seen_hashes']))

## 5. Train
Resumes every model from its latest checkpoint and saves to Drive and Hugging Face after each one. If the runtime disconnects, reconnect and run cells 1-5 again.

Out-of-memory: lower `--batch-size` to 32. On an L4 use `--batch-size 16 --block-size 512`.

In [ ]:
!python scripts/colab_runner.py \
    --drive-root "/content/drive/MyDrive/Helios" \
    --target-tokens 2000000000 \
    --batch-size 64 \
    --block-size 1024 \
    --slice-size 4000

## 6. Progress check

In [ ]:
import json
for t in ('nano','alpha','beta','core','pro','flagship'):
    m = DRIVE_ROOT/'checkpoints'/f'{t}_continuous'/'latest'/'manifest.json'
    if m.exists():
        d = json.load(open(m)); print(f"{t:9s} step={d.get('step', d.get('total_steps'))} loss={d.get('loss', d.get('final_loss'))}")
    else:
        print(f"{t:9s} not started")
ledger = json.load(open(DRIVE_ROOT/'manifests'/'curriculum_ledger.json'))
print("Unique records trained:", len(ledger['seen_hashes']), "| cursor:", ledger['category_cursors']['all'])